# 18 · Advanced Aggregation

Beyond `COUNT`/`SUM`/`AVG`:
- `COUNT(DISTINCT ...)`
- `GROUP_CONCAT` (string aggregation)
- the `FILTER (WHERE ...)` clause — per-aggregate conditions
- conditional aggregation (pivoting rows into columns)
- subtotals & grand totals (emulating `ROLLUP`, which SQLite lacks)

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `COUNT(DISTINCT ...)` and `GROUP_CONCAT`
How many *distinct* products each category contains, plus a comma-joined list of their names:

In [ ]:
%%sql
SELECT c.category_name,
       COUNT(DISTINCT p.product_id) AS num_products,
       GROUP_CONCAT(p.product_name, ', ') AS product_list
FROM categories c
JOIN products p ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY c.category_name;

## The `FILTER` clause
`FILTER (WHERE ...)` restricts which rows feed a *single* aggregate — cleaner
than stuffing `CASE` inside every function. Per employee: total orders vs how
many were completed vs cancelled, in one pass:

In [ ]:
%%sql
SELECT e.first_name,
       COUNT(o.order_id)                                   AS total_orders,
       COUNT(*) FILTER (WHERE o.status = 'completed')      AS completed,
       COUNT(*) FILTER (WHERE o.status = 'cancelled')      AS cancelled,
       COUNT(DISTINCT o.customer_id)                       AS unique_customers
FROM employees e
LEFT JOIN orders o ON o.employee_id = e.employee_id
GROUP BY e.employee_id, e.first_name
ORDER BY total_orders DESC;

## Conditional aggregation (pivot)
The portable equivalent of `FILTER`, and the standard way to *pivot* rows into
columns: put `CASE` inside `SUM`. Revenue per category, split by order status:

In [ ]:
%%sql
SELECT c.category_name,
       ROUND(SUM(CASE WHEN o.status = 'completed' THEN oi.quantity * oi.unit_price ELSE 0 END), 2) AS completed_rev,
       ROUND(SUM(CASE WHEN o.status = 'pending'   THEN oi.quantity * oi.unit_price ELSE 0 END), 2) AS pending_rev
FROM order_items oi
JOIN products p  ON oi.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
JOIN orders o    ON oi.order_id = o.order_id
GROUP BY c.category_name
ORDER BY completed_rev DESC;

## Subtotals + grand total (emulating `ROLLUP`)
Postgres/MySQL have `GROUP BY ... WITH ROLLUP` / `GROUPING SETS`. SQLite doesn't,
so you `UNION ALL` a grand-total row. (Knowing the pattern matters when you move
between databases.)

In [ ]:
%%sql
WITH per_cat AS (
    SELECT c.category_name AS category,
           ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
    FROM order_items oi
    JOIN products p   ON oi.product_id = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY c.category_name
),
with_total AS (
    SELECT category, revenue, 0 AS sort_key FROM per_cat
    UNION ALL
    SELECT 'ALL CATEGORIES', ROUND(SUM(revenue), 2), 1 FROM per_cat
)
SELECT category, revenue
FROM with_total
ORDER BY sort_key, revenue DESC;

## Practice

**✏️ Exercise 1.** For each country, show the number of customers and a GROUP_CONCAT of their first names.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT country, COUNT(*) AS customers, GROUP_CONCAT(first_name, ', ') AS names
FROM customers
GROUP BY country
ORDER BY customers DESC;

**✏️ Exercise 2.** Using FILTER, for each customer show total orders and how many are still pending.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT cu.first_name,
       COUNT(o.order_id) AS orders,
       COUNT(*) FILTER (WHERE o.status = 'pending') AS pending
FROM customers cu
LEFT JOIN orders o ON o.customer_id = cu.customer_id
GROUP BY cu.customer_id, cu.first_name
ORDER BY orders DESC;

### ✅ Recap
`COUNT(DISTINCT)`, `GROUP_CONCAT`, and `FILTER` sharpen aggregation; `CASE`
inside aggregates pivots data; and `UNION ALL` gives you subtotals/grand totals
where `ROLLUP` isn't available.

**Next:** `19_advanced_window_functions.ipynb`.